# Example 4. Use Sweep to automate the execution
## About this example: Sweep class

Sweep allows you to automatically create input files with different values of specific variables.

To work with a sweep, instantiate a Sweep class object as

```python
my_sweep = nextnanopy.Sweep(sweep_variables, path_to_inputfile)
```

Here sweep_variables should be a dict, where keys are the names of variables and the values are iterable objects of values. (ideally lists)

```python
sweep_variables = {'varname1':[val0,val1,val2...],'varname2':[val10,val11,val12...]...}
```

The name of variables to sweep should coincide with the names of variables in input file (see Example1).

In this example we sweep the geometry of a **double quantum well** (`DoubleQuantumWell_6nm_demo.nnp`): the width of the two GaAs wells (`QW_WIDTH`) and the separation between them (`QW_SEPARATION`).

In [1]:
import nextnanopy as nn

path = r'..\..\templates\input files\DoubleQuantumWell_6nm_demo.nnp'
my_sweep = nn.Sweep({'QW_WIDTH':[4.0, 6.0], 'QW_SEPARATION':[2.0, 4.0, 6.0]}, path)
print(my_sweep.input_file)

InputFile
fullpath: ..\..\templates\input files\DoubleQuantumWell_6nm_demo.nnp
Input variables: 7 elements
	$QW_WIDTH = 6.0 # widths of both quantum wells (DisplayUnit:nm)
	$QW_SEPARATION = 4.0 # separation of the QWs        (RangeOfValues:From=1.0,To=14.0,Step=1.0)(ListOfValues:1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10,15,20,30) (DisplayUnit:nm) (HighlightInUserInterface)
	$middle = 28.0 # middle of the structure (DisplayUnit:nm) (DoNotShowInUserInterface)
	$QW1_min = $middle - $QW_SEPARATION / 2 - $QW_WIDTH # minimum coordinate of quantum well #1 (DisplayUnit:nm) (DoNotShowInUserInterface)
	$QW1_max = $middle - $QW_SEPARATION / 2 # maximum coordinate of quantum well #1 (DisplayUnit:nm) (DoNotShowInUserInterface)
	$QW2_min = $middle + $QW_SEPARATION / 2 # minimum coordinate of quantum well #2 (DisplayUnit:nm) (DoNotShowInUserInterface)
	$QW2_max = $middle + $QW_SEPARATION / 2 + $QW_WIDTH # maximum coordinate of quantum well #2 (DisplayUnit:nm) (DoNotShowInUserInterface)


To save a sweep, use `Sweep.save()`. Passing `temp=True` writes the generated input files into a temporary directory that is created on first use and removed when the Python process exits, so nothing is left behind next to the prototype input file.

In [2]:
my_sweep.save(temp=True)

After saving with `temp=True`, one input file per combination of the sweep variables is created inside a temporary directory. One can access info about the generated input files via

In [3]:
my_sweep.input_files

[InputFile
 fullpath: C:\Users\Heorhii\AppData\Local\Temp\tmpvbrmt03c\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_2.0_.nnp
 Input variables: 7 elements
 	$QW_WIDTH = 4.0 # THIS VARIABLE IS UNDER SWEEP
 	$QW_SEPARATION = 2.0 # THIS VARIABLE IS UNDER SWEEP
 	$middle = 28.0 # middle of the structure (DisplayUnit:nm) (DoNotShowInUserInterface)
 	$QW1_min = $middle - $QW_SEPARATION / 2 - $QW_WIDTH # minimum coordinate of quantum well #1 (DisplayUnit:nm) (DoNotShowInUserInterface)
 	$QW1_max = $middle - $QW_SEPARATION / 2 # maximum coordinate of quantum well #1 (DisplayUnit:nm) (DoNotShowInUserInterface)
 	$QW2_min = $middle + $QW_SEPARATION / 2 # minimum coordinate of quantum well #2 (DisplayUnit:nm) (DoNotShowInUserInterface)
 	$QW2_max = $middle + $QW_SEPARATION / 2 + $QW_WIDTH # maximum coordinate of quantum well #2 (DisplayUnit:nm) (DoNotShowInUserInterface),
 InputFile
 fullpath: C:\Users\Heorhii\AppData\Local\Temp\tmpvbrmt03c\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPA

To execute the sweep, run 
```python
Sweep.execute()
```

Under execution, a few things will happen.
1. The directory inputfilename_sweep_var1_var2_...varn will be created in the output directory.
2. The sweep.info file with the information of the sweep is saved there.
3. Input files from sweep.input_files are executed and output is saved to the mentioned folder.

Sweep.execute() has 5 optional parameters + can take any parameter accepted by nextnanopy.InputFile.execute().

1. delete_input_files: bool, optional

When set to `True`, input_files will be deleted after execution. Default is `False`.

2. overwrite: bool, optional

When set to `True`, the output will overwrite the old output. When `False`, execution will create new output folder (with the unique name, created by adding an integer to the foldername). Default is `False`.

3. show_log: bool, optional

When set to `True`, the simulation log is displayed in the console, while `False` suppresses the log. Default is `True`. Note that the log file is always saved in the output folders regardless of this option.

4. convergenceCheck: bool, optional

When set to `True`, nextnanopy scans the log file of the simulation performed and check whether the solution has converged. If it did not converge, nextnanopy warns you and ask if you want to proceed with postprocessing. Note that non-converged solutions are not reliable and further calculation and/or visualization from them do not make much sense. Default is `False`.

5.  parallel_limit: int, optional

number of simulation to run simultaneously. Especially useful for simple simulations which might be more efficiently run in parallel. Be aware that some nextnano solvers parallelize computations internally in threads (controlled by --threads in nextnanopy config). To avoid unexpected behaviour and not desirable decrease of simulation speed use the rule: parallel_limit*threads<= number of physical cores of the machine
default parallel_limit =  1 

**kwargs
Any other parameter accepted by nextnanopy.InputFile.execute() e.g. exe, license, database, outputdirectory

Example of the simulation in parallel (2 Input files at a time)


In [4]:
my_sweep.execute(overwrite=True, show_log=False, parallel_limit=2)


Remaining simulations in the queue:  5

Remaining simulations in the queue:  4

Remaining simulations in the queue:  3

Remaining simulations in the queue:  2

Remaining simulations in the queue:  1

Remaining simulations in the queue:  0

Waiting queue is empty, all execution and logging are finished


Example of the simulation in sequence (if you want to run the sweep which was already executed, save it one more time)

In [5]:
my_sweep.save(temp=True, delete_old_files=True)
my_sweep.execute(overwrite=True, show_log=False, parallel_limit=1)


Executing simulations [1/6]...

Executing simulations [2/6]...

Executing simulations [3/6]...

Executing simulations [4/6]...

Executing simulations [5/6]...

Executing simulations [6/6]...


## Match output folders to the sweep parameters

Once the sweep has been executed, `my_sweep.sweep_output_infodict` records, for every simulation, the output folder it was written to together with the combination of sweep variables that produced it. It is a `DictList` keyed by the output-folder path, so iterating it recovers the correspondence in either direction — look up which folder holds the result for a given set of parameters, or read off which parameters a folder was run with. nextnanopy also writes this same mapping to `sweep_infodict.json` inside the sweep output directory.

In [7]:
for output_folder, parameters in my_sweep.sweep_output_infodict.items():
    print(f"{parameters} -> {output_folder}")

{'QW_WIDTH': 4.0, 'QW_SEPARATION': 2.0} -> C:\Users\Heorhii\Documents\nextnano\Output\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_2.0_
{'QW_WIDTH': 4.0, 'QW_SEPARATION': 4.0} -> C:\Users\Heorhii\Documents\nextnano\Output\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_4.0_
{'QW_WIDTH': 4.0, 'QW_SEPARATION': 6.0} -> C:\Users\Heorhii\Documents\nextnano\Output\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_6.0_
{'QW_WIDTH': 6.0, 'QW_SEPARATION': 2.0} -> C:\Users\Heorhii\Documents\nextnano\Output\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\DoubleQuantumWell_6nm_demo__QW_WIDTH_6.0_QW_SEPARATION_2.0_
{'QW_WIDTH': 6.0, 'QW_SEPARATION': 4.0} -> C:\Users\Heorhii\Documents\nextnano\Output\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\DoubleQuantumWell_6nm_demo__QW_WIDTH_6.0

## Restore the mapping in another process

The correspondence is not only kept in memory: `Sweep.execute()` writes it to `sweep_infodict.json` inside the sweep output directory (`my_sweep.sweep_output_directory`). Because it lives on disk, the mapping outlives the Python session — a separate script that never built the `Sweep` object can recover it from the folder alone. Below we reuse `my_sweep.sweep_output_directory` for convenience; in a fresh process you would simply hard-code that directory path.

nextnanopy reads it back with `DataFolder.read_sweep_infodict()`:

In [8]:
sweep_dir = my_sweep.sweep_output_directory

restored = nn.DataFolder(sweep_dir).read_sweep_infodict()
restored

{'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_2.0_': {'QW_WIDTH': 4.0,
  'QW_SEPARATION': 2.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_4.0_': {'QW_WIDTH': 4.0,
  'QW_SEPARATION': 4.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_6.0_': {'QW_WIDTH': 4.0,
  'QW_SEPARATION': 6.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_6.0_QW_SEPARATION_2.0_': {'QW_WIDTH': 6.0,
  'QW_SEPARATION': 2.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW

The file is plain JSON, so any code can read it back without nextnanopy:

In [9]:
import json
import os

with open(os.path.join(sweep_dir, "sweep_infodict.json")) as file:
    restored = json.load(file)
restored

{'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_2.0_': {'QW_WIDTH': 4.0,
  'QW_SEPARATION': 2.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_4.0_': {'QW_WIDTH': 4.0,
  'QW_SEPARATION': 4.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_4.0_QW_SEPARATION_6.0_': {'QW_WIDTH': 4.0,
  'QW_SEPARATION': 6.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW_WIDTH_6.0_QW_SEPARATION_2.0_': {'QW_WIDTH': 6.0,
  'QW_SEPARATION': 2.0},
 'C:\\Users\\Heorhii\\Documents\\nextnano\\Output\\DoubleQuantumWell_6nm_demo_sweep__QW_WIDTH__QW_SEPARATION\\DoubleQuantumWell_6nm_demo__QW



Please contact python@nextnano.com for any issues with this document.